# 🚗 Regression Practice Lab (Auto MPG)

This lab extends the material in *Regression — Lecture 4* and the companion `Regression_demo.ipynb`. You'll apply the same workflow to a new automotive dataset, adding deeper diagnostics (correlations, multicollinearity) and modern hyperparameter tuning with Optuna.

## 🎯 Learning Goals
- Audit and prepare an automotive efficiency dataset for multiple linear regression.
- Engineer informative features and quantify pairwise correlations.
- Diagnose multicollinearity via Variance Inflation Factors (VIF).
- Fit and compare baseline/regularized linear models (OLS, Ridge, Lasso).
- Run automated hyperparameter optimization with **Optuna** on a tree-based regressor.
- Evaluate residual behavior to surface model limitations.

## 📦 Dataset
We'll work with the classic **Auto MPG** dataset (fuel economy data for 1970s–80s vehicles) curated by the U.S. Environmental Protection Agency and hosted in the open Seaborn repository:
`https://raw.githubusercontent.com/mwaskom/seaborn-data/master/mpg.csv`.

Target variable: `mpg` (miles per gallon).
Key predictors: engine displacement, horsepower, weight, acceleration, cylinder count, model year, and origin region.

---
# Part 1 · Setup & Data Audit
Reproduce the lecture's data-understanding steps: load, inspect schema & missingness, and visualize target relationships.

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from pathlib import Path

from sklearn.model_selection import train_test_split, cross_validate, KFold, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor

from statsmodels.stats.outliers_influence import variance_inflation_factor
import optuna

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (10, 6)

DATA_URL = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/mpg.csv"

# Tip: run `pip install optuna statsmodels` if these imports fail in your environment.

## Task 1 · Load & peek at the data
1. Read the CSV into a DataFrame `df` (optionally cache it to `DATA_CACHE_PATH`).
2. Show its shape and first five rows.
3. Confirm `mpg` is numeric and inspect its basic stats.

In [ ]:
raise NotImplementedError("Load data from DATA_URL (optionally caching) and inspect shape/head/describe.")

<details>
<summary>Need a nudge?</summary>
- Use `pd.read_csv(DATA_URL)` or read from the cached file if it exists.
- After loading, call `df.shape`, `df.head()` and `df['mpg'].describe()`.
</details>

## Task 2 · Missingness & numeric summary
Following the regression checklist, compute:
- A Series of missing counts per column (sorted descending).
- Descriptive stats (`describe()`) for numeric predictors.
Document any large gaps or extreme ranges you should address before modeling.

In [ ]:
# Pseudocode steps:
# 1. Compute missing counts with df.isna().sum(), sort descending.
# 2. Call describe() on numeric subset (df.select_dtypes).
# 3. Record findings (e.g., via Markdown cell) about gaps/outliers.
raise NotImplementedError("Produce missing-value counts and numeric describe table(s).")

## Task 3 · Visual diagnostics
Create exploratory plots to inspect relationships with `mpg`. Suggested options:
- Scatterplot `horsepower` vs `mpg` with a regression line.
- Pair `weight` and `acceleration`, colored by `origin` or `cylinders`.
- Box/violin plot of `mpg` grouped by `origin` (or binned `cylinders`).

In [ ]:
# Pseudocode steps:
# 1. Create regression scatter for horsepower vs mpg (sns.regplot or lmplot).
# 2. Build 2D scatter colored by origin/cylinders.
# 3. Add box/violin comparing mpg by origin.
# 4. Add at least one more plot that surfaces another pattern.
raise NotImplementedError("Visualize key relationships per the suggestions (can use seaborn).")

## Task 4 · Feature engineering
Following best practices from the lecture, create derived columns that capture efficiency and power ratios, for example:
- `power_to_weight = horsepower / weight`
- `displacement_per_cylinder = displacement / cylinders`
- `acceleration_per_weight = acceleration / weight`
Guard against division-by-zero or missing values before adding them to `df`. 

In [ ]:
# Pseudocode steps:
# 1. Work on a copy of df to avoid chained assignment warnings.
# 2. Replace zero denominators with np.nan before division.
# 3. Derive power_to_weight, displacement_per_cylinder, acceleration_per_weight.
# 4. Optionally engineer more ratios/features you believe help mpg.
raise NotImplementedError("Create power_to_weight, displacement_per_cylinder, acceleration_per_weight with safe operations.")

## Task 5 · Correlations & multicollinearity
1. Build a correlation heatmap (numeric columns only) highlighting absolute correlations > 0.7.
2. Compute VIF values for your numeric predictors (including engineered features).
3. Note which predictors may cause multicollinearity issues and how you'd address them.

In [ ]:
# Pseudocode steps:
# 1. Compute df.corr(numeric_only=True); mask upper triangle when plotting heatmap.
# 2. Plot heatmap with annotations for clarity.
# 3. Assemble VIF table: select numeric predictors (drop mpg), dropna, loop variance_inflation_factor.
# 4. Interpret which features require mitigation.
raise NotImplementedError("Compute correlations, plot heatmap, and tabulate VIF for numeric columns.")

---
# Part 2 · Modeling
Now that the dataset is vetted, reproduce the modeling stack from the lecture with additional regularization and hyperparameter search.

## Task 6 · Define features & train/test split
- Set `y = df['mpg']`.
- Build `X` from the remaining columns (consider dropping the free-text `name`).
- Identify `numeric_features` vs `categorical_features` (hint: treat `origin` and `cylinders` as categorical).
- Split into train/test (80/20) with `random_state=42`. 

In [ ]:
# Pseudocode steps:
# 1. Drop rows with missing target and the free-text name column.
# 2. Cast origin/cylinders as categorical (e.g., str) so they encode later.
# 3. Split features/target, capture numeric vs categorical feature lists.
# 4. Run train_test_split (80/20, random_state=42) and print shapes.
raise NotImplementedError("Define target/features, set numeric/categorical lists, and perform train/test split.")

## Task 7 · Build the preprocessing + linear regression pipeline
- Numeric transformer: `SimpleImputer(strategy=\"median\")` ➜ `StandardScaler`.
- Categorical transformer: `SimpleImputer(strategy=\"most_frequent\")` ➜ `OneHotEncoder(handle_unknown=\"ignore\")`.
- Combine with `ColumnTransformer` and wrap in `Pipeline([(\"preprocess\", ...), (\"model\", LinearRegression())])`.

In [ ]:
# Pseudocode steps:
# 1. Build numeric pipeline (median imputer -> scaler).
# 2. Build categorical pipeline (most_frequent imputer -> OneHotEncoder).
# 3. Combine with ColumnTransformer mapping lists from prior task.
# 4. Wrap preprocess + LinearRegression into Pipeline.
raise NotImplementedError("Create preprocess ColumnTransformer and baseline_pipeline with LinearRegression.")

## Task 8 · Fit & evaluate the baseline
1. Fit the pipeline on the training data.
2. Report training $R^2$, test $R^2$, and test RMSE (root mean squared error).
3. Store the metrics in a neat dict/DataFrame for future comparison.

In [ ]:
# Pseudocode steps:
# 1. Fit baseline_pipeline on training split.
# 2. Generate train/test predictions.
# 3. Compute train R^2, test R^2, and test RMSE.
# 4. Store metrics in a tidy structure (dict/Series/DataFrame).
raise NotImplementedError("Train baseline_pipeline and calculate train/test R^2 plus test RMSE.")

## Task 9 · Regularization with cross-validation
Compare Ridge and Lasso using 5-fold CV:
- Reuse the preprocessing steps but replace the estimator with `Ridge()` / `Lasso()`.
- Tune `alpha` over `np.logspace(-3, 2, 10)` using `GridSearchCV` or `cross_validate` with manual looping.
- Report the best alpha, mean CV RMSE, and holdout metrics for each model.

In [ ]:
# Pseudocode steps:
# 1. Clone preprocess pipeline, append Ridge estimator, wrap in GridSearchCV.
# 2. Repeat for Lasso (increase max_iter for convergence).
# 3. Evaluate best params on test set, compute RMSE.
# 4. Summarize CV vs test metrics for both models.
raise NotImplementedError("Tune alpha for Ridge/Lasso and summarize CV + test performance.")

In [ ]:
# Pseudocode steps:
# 1. Extract coefficients: ols_coef, ridge_coef, lasso_coef from fitted models.
# 2. Get feature names from preprocessor (numeric + categorical encoded names).
# 3. Create DataFrame with columns: Feature, OLS, Ridge, Lasso.
# 4. Sort by absolute OLS coefficient to find most influential features.
# 5. Plot horizontal bar chart comparing coefficients across models.
# 6. Count and visualize "active" features (threshold = 0.01 or 0.001).
# 7. Write interpretation: Which features matter? What did Lasso eliminate?
raise NotImplementedError("Extract and compare coefficients from OLS, Ridge, and Lasso models.")

## Task 9a · Linear model coefficient analysis
Analyze and compare the coefficients from your trained linear models (OLS baseline, Ridge, Lasso):
- Extract the `.coef_` attribute from each fitted model.
- Get feature names from the preprocessing step (use `.get_feature_names_out()` on transformers).
- Create a DataFrame showing coefficients side-by-side for all three models.
- Visualize: (a) Top 10 features by absolute coefficient value, (b) Count of "active" features (|coef| > threshold).
- Interpret: Which features have the strongest influence? How did regularization affect feature selection?

## Task 10 · Tree-based models with Optuna optimization
Use **Optuna** to optimize both `DecisionTreeRegressor` and `RandomForestRegressor`:
1. **Decision Tree**: Define objective function sampling max depth, min samples split/leaf, and max features.
2. **Random Forest**: Define objective sampling n_estimators, max depth, min samples split/leaf, and max features.
3. Use 5-fold CV (e.g., `KFold(random_state=42, shuffle=True)`) inside objectives to compute RMSE.
4. Run at least 20-50 trials for each, capture best hyperparameters, and evaluate on test set.

In [ ]:
# Pseudocode steps:
# 1. Define dt_objective(trial) with DecisionTreeRegressor + trial-suggested hyperparameters.
# 2. Define rf_objective(trial) with RandomForestRegressor + trial-suggested hyperparameters.
# 3. Run cross_validate (5-fold) scoring negative RMSE; return mean score for each.
# 4. Optimize studies with ~20-50 trials each, retrieve best params.
# 5. Fit final pipelines on train set and evaluate RMSE on test set.
raise NotImplementedError("Optimize DT and RF with Optuna, report best hyperparameters and test metrics.")

In [ ]:
# Pseudocode steps:
# 1. Extract feature_importances_ from fitted DecisionTree and RandomForest models.
# 2. Use same feature_names list from Task 9a.
# 3. Create DataFrame with columns: Feature, Decision Tree, Random Forest.
# 4. Sort by Random Forest importance (typically more stable).
# 5. Plot: (a) horizontal bar chart for top 10, (b) scatter DT vs RF with diagonal line.
# 6. Calculate correlation between DT and RF importances.
# 7. Write interpretation: Top predictors? RF stability? Agreement with linear coefficients?
raise NotImplementedError("Extract and compare feature importances from Decision Tree and Random Forest.")

## Task 10a · Tree model feature importance analysis
Analyze and compare feature importances from your trained Decision Tree and Random Forest models:
- Extract the `.feature_importances_` attribute from each fitted tree model.
- Use the same feature names from the preprocessing step.
- Create a DataFrame comparing importances side-by-side for DT and RF.
- Visualize: (a) Top 10 features comparison (horizontal bars), (b) Scatter plot of DT vs RF importances.
- Interpret: Which features are most important? How stable is DT vs RF? Do tree models agree with linear model coefficients?

## Task 11 · Residual diagnostics
Using your preferred model (best performing overall), plot residuals vs predicted values on the test set. Add a horizontal zero line and comment on heteroscedasticity or outliers you observe.

In [ ]:
# Pseudocode steps:
# 1. Pick the model with best generalization (e.g., lowest test RMSE).
# 2. Predict on X_test and compute residuals = y_test - y_pred.
# 3. Plot residuals vs predictions, add horizontal zero reference.
# 4. Inspect for heteroscedasticity/outliers; jot reflections in Markdown.
raise NotImplementedError("Generate residual plot for test set predictions and interpret patterns.")

---
## 📝 Wrap-up Reflection
Answer in Markdown:
1. Which predictors (or engineered features) most strongly influenced fuel economy (`mpg`)? Reference coefficients or feature importances.
2. Did regularization or tree-based tuning improve generalization versus the baseline? Cite metrics.
3. How would you further address multicollinearity, heteroscedasticity, or feature selection if deploying this model?